### Objective
0. Run after neur5.ipynb
1. Is there a context axis?

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os, pickle
import scipy.stats as stats, scipy.io as sio
from sklearn.decomposition import PCA


plot settings

In [2]:
# sns.set(context='paper')

# keep text editable in svg
plt.rcParams['svg.fonttype'] = 'none'

import matplotlib as mpl
# push ticks inward
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
# remove top and right splines
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

# %matplotlib widget
%matplotlib inline


variables & helpers (from neur4)

In [ ]:
conds, bounds, epoch = ['curv', 'base', 'flat'], [.25, .5, .75], 'stim'

with open(f'../../outputs/decoding_matrices/{epoch}_pts_data.pkl', 'rb') as f: pts_data = pickle.load(f)
print(f'pts_data.keys(): {pts_data.keys()}')
print(f'pts_data[12].keys(): {pts_data[12].keys()}')
print(f"pts_data[12]['psychopy'].shape (trials, cols): {pts_data[12]['psychopy'].shape}")
print(f"pts_data[12]['meanFRs'].shape (trials, neurons): {pts_data[12]['meanFRs'].shape}")
print(f"pts_data[12]['n_neurs']: {pts_data[12]['n_neurs']}\n")
    
with open(f'../../outputs/decoding_matrices/{epoch}_cond_pts_FRs.pkl', 'rb') as f: cond_pts_FRs = pickle.load(f)
print(f"cond_pts_FRs.keys(): {cond_pts_FRs.keys()}")
print(f"cond_pts_FRs['curv'].keys(): {cond_pts_FRs['curv'].keys()}")
print(f"cond_pts_FRs['curv'][12].shape (unique_stims, neurons): {cond_pts_FRs['curv'][12].shape}\n")

with open(f'../../outputs/decoding_matrices/{epoch}_cond2stims.pkl', 'rb') as f: cond2stims = pickle.load(f)
print(f"cond2stims.keys(): {cond2stims.keys()}")
print(f"cond2stims['curv'].shape (unique_stims,): {cond2stims['curv'].shape}\n")

with open(f'../../outputs/decoding_matrices/{epoch}_cond2stimXneur.pkl', 'rb') as f: cond2stimXneur = pickle.load(f)
print(f"cond2stimXneur.keys(): {cond2stimXneur.keys()}")
print(f"cond2stimXneur['curv'].shape (unique_stims, neurons): {cond2stimXneur['curv'].shape}\n")

with open(f'../../outputs/decoding_matrices/{epoch}_cond_decode_X.pkl', 'rb') as f: cond_decode_X = pickle.load(f)
with open(f'../../outputs/decoding_matrices/{epoch}_cond_decode_y.pkl', 'rb') as f: cond_decode_y = pickle.load(f)
print(f'cond_decode_X.keys(): {cond_decode_X.keys()}')
print(f'cond_decode_X["curv"].shape (trials, neurons): {cond_decode_X["curv"].shape}')
print(f'cond_decode_y.keys(): {cond_decode_y.keys()}')
print(f'cond_decode_y["curv"].shape (trials,): {cond_decode_y["curv"].shape}\n')

### 1. Is there a context axis?

In [6]:
# merge all data
all_stimXneur = np.concatenate(list(cond2stimXneur.values()), axis=0)
print(f'all_stimXneur.shape (all_stims, neurons): {all_stimXneur.shape}')

all_stimXneur.shape (all_stims, neurons): (50, 57)


In [7]:
pts_data[12]['psychopy'].columns

Index(['trial_key', 'thisN', 'thisTrialN', 'thisRepN', 'blockN', 'run',
       'condition', 'stim_file_pos', 'true_stim', 'noise_pos',
       ...
       'stim_boundary_aligned', 'resp_boundary_aligned', 'rank_stim',
       'rank_resp', 'baseline_dur', 'stim_dur', 'delay_dur', 'task_dur',
       'anticipation_dur', 'feedback_dur'],
      dtype='str', length=159)

In [10]:
patients = sorted(pts_data.keys())

In [11]:
# per cond, create X & y for decoding
cond_decode_X, cond_decode_y = {}, {}

for cond in conds:

    # loop over patients to hstack neurons
    pt_X, cond_ids_check = [], []
    for pt in patients:   
        
        pt_psych_df = pt_data[pt]['psychopy'].sort_values('trial_key').reset_index(drop=True)
        cond_ids = pt_psych_df['condition'] == cond
        cond_pt_psych_df = pt_psych_df[cond_ids]
        assert pt_psych_df['trial_key'].is_monotonic_increasing
        assert cond_ids.sum() == 80

        # obtain X = FR_mtx (trial, neur)
        pt_X.append(pt_data[pt]['meanFRs'][cond_ids])

    cond_decode_X[cond] = np.hstack(pt_X)

    # obtain y = class labels
    cond_decode_y[cond] = y = (cond_pt_psych_df['stim_boundary_aligned'] > 0).astype(int).values
    assert cond_decode_y[cond].sum() == 40 # 40 1s, 40 0s

print(cond_decode_X.keys())
assert cond_decode_X['curv'].shape == cond_decode_X['base'].shape == cond_decode_X['flat'].shape
print(f'All matrices have the same (trials, neurs); example: {cond_decode_X["curv"].shape}')


NameError: name 'pt_data' is not defined